# Parte 3: Validación de hipótesis de cancelación (ejercicios 17 al 20)

In [0]:
%sql
-- Configuración previa
USE CATALOG cursodataacademy;
USE SCHEMA hotel_db;

## Ejercicio 17. Hipótesis 1: ¿Los hijos aseguran el viaje?

In [0]:
%sql
WITH reservas_clasificadas AS (
    SELECT
        is_canceled,

        CASE
            WHEN COALESCE(children, 0) > 0
              OR COALESCE(babies, 0) > 0
                THEN 'Con Hijos'
            ELSE 'Sin Hijos'
        END AS categoria_hijos

    FROM tb_reservas
)

SELECT
    categoria_hijos,
    COUNT(*) AS total_reservas,
    SUM(is_canceled) AS total_cancelaciones,

    ROUND(
        AVG(is_canceled) * 100,
        2
    ) AS tasa_cancelacion_pct

FROM reservas_clasificadas

GROUP BY categoria_hijos

ORDER BY tasa_cancelacion_pct DESC;

**Conclusión:** Las reservas del segmento "Sin Hijos" presentan la mayor tasa de cancelación, con un 37.22 %, frente al 34.92 % del otro segmento. Esto permite evaluar si la presencia de hijos está asociada con una mayor estabilidad de las reservas.

## Ejercicio 18. Hipótesis 2: Impacto de las solicitudes especiales

In [0]:
%sql
SELECT
    total_of_special_requests AS cantidad_solicitudes,

    COUNT(*) AS total_reservas,

    SUM(is_canceled) AS total_cancelaciones,

    ROUND(
        AVG(is_canceled) * 100,
        2
    ) AS tasa_cancelacion_pct

FROM tb_reservas

GROUP BY total_of_special_requests

ORDER BY cantidad_solicitudes ASC;

**Conclusión:** Las reservas con 0 solicitudes especiales presentan una tasa de cancelación del 47.72 %. Los resultados muestran una tendencia decreciente, lo que permite evaluar la relación entre las solicitudes especiales y el compromiso de los huéspedes.

## Ejercicio 19. Hipótesis 3: ¿Cambiar la reserva es señal de compromiso?

In [0]:
%sql
WITH reservas_clasificadas AS (
    SELECT
        is_canceled,

        CASE
            WHEN booking_changes = 0
                THEN 'Sin Cambios'

            WHEN booking_changes BETWEEN 1 AND 3
                THEN '1 a 3 cambios'

            ELSE 'Más de 3 cambios'
        END AS categoria_cambios

    FROM tb_reservas
    WHERE booking_changes IS NOT NULL
)

SELECT
    categoria_cambios,

    COUNT(*) AS total_reservas,

    SUM(is_canceled) AS total_cancelaciones,

    ROUND(
        AVG(is_canceled) * 100,
        2
    ) AS tasa_cancelacion_pct

FROM reservas_clasificadas

GROUP BY categoria_cambios

ORDER BY tasa_cancelacion_pct ASC;

**Conclusión:** Las reservas con "1 a 3 cambios" presentan la menor tasa de cancelación, equivalente al 15.59 %. Este resultado sugiere que el número de modificaciones podría estar asociado con el nivel de compromiso de los huéspedes.

## Ejercicio 20. Hipótesis 4: El factor precio (ADR)

In [0]:
%sql
SELECT
    hotel,

    COUNT(*) AS total_cancelaciones,

    ROUND(
        AVG(adr),
        2
    ) AS precio_promedio_cancelaciones

FROM tb_reservas

WHERE is_canceled = 1

GROUP BY hotel

ORDER BY precio_promedio_cancelaciones DESC;

**Conclusión:** El Resort Hotel presenta el mayor precio promedio entre las reservas canceladas, con un ADR de US$ 105.79, frente a US$ 104.69 del otro hotel. Esto permite identificar en qué segmento las cancelaciones se asocian con tarifas promedio más elevadas.